# 记忆治理策略

## 1、消息裁剪

In [2]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("CLOSEAI_API_KEY"),
    base_url=os.getenv("CLOSEAI_BASE_URL")
)

In [3]:
from langchain_core.messages import HumanMessage
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig
from typing import Any

@before_model
def trim_messages(state:AgentState,runtime:Runtime) -> dict[str, Any] | None:

    messages = state["messages"]

    if len(messages) <= 3:
        return None

    first_message = messages[0]
    # 如果有偶数条消息，则取最近的3条消息；如果有奇数条消息，则取最近的4条消息
    recent_messages = messages[-3:] if len(messages) % 2 == 0 else messages[-4:]

    new_messages = [first_message] + recent_messages

    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *new_messages
        ],
    }

agent = create_agent(
    model=model,
    middleware=[trim_messages],
    checkpointer=InMemorySaver(),
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": [HumanMessage("你好，我是老王")]}, config)
agent.invoke({"messages": [HumanMessage("从现在起，你叫小王")]}, config)
agent.invoke({"messages": [HumanMessage("今天天气不错")]}, config)
final_response = agent.invoke({"messages": [HumanMessage("告诉我，你是谁？我是谁？")]}, config)

for msg in final_response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好，我是老王
================================== Ai Message ==================================

好的，从现在起你可以叫我“小王”。  
我在，老王，有什么事尽管说。
================================ Human Message =================================

今天天气不错
================================== Ai Message ==================================

是啊，天气不错的话，心情也容易跟着好起来。你今天打算出去走走，还是在家休息？
================================ Human Message =================================

告诉我，你是谁？我是谁？
================================== Ai Message ==================================

我是一个AI助手，可以帮你回答问题、写作、翻译、总结、聊天。  

你是“老王”——至少按照你刚才告诉我的信息是这样。  
如果你愿意，也可以告诉我你希望我怎么称呼你。


## 2、消息删除

In [8]:
from langchain.messages import RemoveMessage
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import after_model
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig


@after_model
def delete_old_messages(state: AgentState, runtime: Runtime) -> dict | None:
    messages = state["messages"]
    # 保持最近的 5 条消息
    if len(messages) > 5:
        # 框架中通常使用 RemoveMessage 来标记删除，并返回更新状态。
        to_delete = len(messages) - 5
        return {"messages": [RemoveMessage(id=m.id) for m in messages[:to_delete]]}
    return None


agent = create_agent(
    model=model,
    middleware=[delete_old_messages],
    checkpointer=InMemorySaver()
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": "你好，我是老王"}, config)
agent.invoke({"messages": "从现在起，你叫小王"}, config)
agent.invoke({"messages": "今天天气不错"}, config)
final_response = agent.invoke({"messages": "告诉我，你是谁？我是谁？"}, config)

for msg in final_response["messages"]:
    msg.pretty_print()

================================== Ai Message ==================================

好的，从现在起你可以叫我小王。  
我会继续帮你，有什么事尽管说。
================================ Human Message =================================

今天天气不错
================================== Ai Message ==================================

是啊，天气不错的时候心情也容易变好。  
你今天有什么安排吗？
================================ Human Message =================================

告诉我，你是谁？我是谁？
================================== Ai Message ==================================

我是小王，一个帮你回答问题、提供建议、一起聊天的 AI 助手。

你是我的对话对象；如果按你刚才给我的称呼，你是和“小王”聊天的人。  
如果你愿意，也可以告诉我你希望我怎么称呼你。


## 3、摘要

In [9]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

model_out = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("CLOSEAI_API_KEY"),
    base_url=os.getenv("CLOSEAI_BASE_URL")
)

In [10]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

model_in = init_chat_model(
    model="gpt-4o-mini",
    model_provider="openai",
    api_key=os.getenv("CLOSEAI_API_KEY"),
    base_url=os.getenv("CLOSEAI_BASE_URL")
)

In [11]:
from langchain.agents.middleware import SummarizationMiddleware
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver


# 创建带摘要中间件的 Agent
agent = create_agent(
    model=model_out,
    tools=[],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model_in,
            trigger=[
                ("tokens", 100),  # 超过 100 tokens 就摘要
            ],
            keep=("messages", 2),
            summary_prompt="对历史消息摘要，消息列表如下\n{messages}",

        )
    ]
)

config = {"configurable": {"thread_id": "1"}}

print("\n进行多轮对话...")
conversations = [
    "我叫张三，是工程师。这里是一段非常长非常长的废话..." * 20, # 强制撑爆 100 tokens
    "请总结一下我的信息"
]

for msg in conversations:
    response = agent.invoke(
        {"messages": [{"role": "user", "content": msg}]},
        config=config
    )
    for msg in response["messages"]:
        msg.pretty_print()
    print("*" * 50)


进行多轮对话...
================================ Human Message =================================

我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...我叫张三，是工程师。这里是一段非常长非常长的废话...
================================== Ai Message ==================================

你好，张三！你是工程师。  
如果你想把这段“非常长非常长的废话”做摘要、提取关键信息、改写成简历自我介绍，或者压缩成一句话，我都可以帮你。
**************************************************
================================ Human Message =================================

Here is a summary of the conversation to date:

这段消息的主要内容是一个名叫张三的工程师重复介绍自己，并且包含了

In [12]:
from rich import print as rprint

final_state = agent.get_state(config)

rprint(final_state)

StateSnapshot(
    values={
        'messages': [
            HumanMessage(
                content='Here is a summary of the conversation to 
date:\n\n这段消息的主要内容是一个名叫张三的工程师重复介绍自己，并且包含了一些冗长的废话。信息未提供具体的实质性内容
，主要是重复性的自我介绍。',
                additional_kwargs={'lc_source': 'summarization'},
                response_metadata={},
                id='93e73869-5a2e-45bf-ae4e-2d7ea14fe4fd'
            ),
            AIMessage(
                content='你好，张三！你是工程师。  
\n如果你想把这段“非常长非常长的废话”做摘要、提取关键信息、改写成简历自我介绍，或者压缩成一句话，我都可以帮你。',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 61,
                        'prompt_tokens': 386,
                        'total_tokens': 447,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': 0,
                            'audio_tokens': 0,
                            'reasoning_tokens': 0,
                            'rejected_prediction_tokens': 0
                        },
                        'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
                        'latency_checkpoint': {
                            'engine_tbt_ms': 7,
                            'engine_ttft_ms': 49,
                            'engine_ttlt_ms': 492,
                            'pre_inference_ms': 73,
                            'service_tbt_ms': 7,
                            'service_ttft_ms': 658,
                            'service_ttlt_ms': 1094,
                            'total_duration_ms': 1030,
                            'user_visible_ttft_ms': 585
                        }
                    },
                    'model_provider': 'openai',
                    'model_name': 'gpt-5.4-mini-2026-03-17',
                    'system_fingerprint': None,
                    'id': 'chatcmpl-Dpyq9zYSrd77pa1r2Zr8rQQLoK9CU',
                    'service_tier': 'default',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--019ebc9e-8afa-7640-9242-996fc93f9759-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 386,
                    'output_tokens': 61,
                    'total_tokens': 447,
                    'input_token_details': {'audio': 0, 'cache_read': 0},
                    'output_token_details': {'audio': 0, 'reasoning': 0}
                }
            ),
            HumanMessage(
                content='请总结一下我的信息',
                additional_kwargs={},
                response_metadata={},
                id='0d9196e4-b400-45d4-bcc5-da630890d6c4'
            ),
            AIMessage(
                content='根据你目前提供的信息，我能总结出的是：\n\n- 你叫 **张三**\n- 你的职业是 **工程师**\n- 
你希望我帮你**总结信息**，而且你前面的内容里有很多**重复和冗长**的表述\n\n如果你愿意，我还可以继续帮你整理成更正式
的版本，比如：\n1. **一句话简介**\n2. **简历版个人信息**\n3. **更简洁的中文总结**',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 108,
                        'prompt_tokens': 136,
                        'total_tokens': 244,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': 0,
                            'audio_tokens': 0,
                            'reasoning_tokens': 0,
                            'rejected_prediction_tokens': 0
                        },
                        'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
                        'latency_checkpoint': {
                            'engine_tbt_ms': 5,
                            'engine_ttft_ms': 36,
                            'engine_ttlt_ms': 613,
                            'pre_inference_ms': 66,
                            'service_tbt_ms': 5,
                 